# EOD Deliverable — Output 2: Tree Age Estimation / Age Proxy
## Python / Earth Engine API translation of `20I_MASTER_PIPELINE_OUTPUT2_TREE_AGE.js`

**Purpose:** reproduce the validated GEE master integration for Output 2 in a Python notebook that a mentor/team member can run and compare against the Code Editor result.

### Scope of 20I (important)
20I is **not raw-imagery-to-final**. It starts from three standardized inputs already produced upstream:
1. Output 1 plantation boundary;
2. Landsat historical clean stack (2001–2018);
3. Sentinel-2 recent regrowth stack (2019–2025).

Upstream scripts 14–19 remain required to rebuild those temporal stacks for a new AOI.

### Locked methodology
- Valid historical Landsat event is retained unless a **valid Sentinel-2 recent event** overrides it.
- Uncertain S2 does **not** erase a valid Landsat event.
- No second MMU is applied to age/event classes; the 0.50 ha MMU belongs to Output 1 boundary.
- Final resolution = **30 m**.
- Six final classes: class 0 + five event-period classes.
- **Class 0 = possible pre-2001 OR no event detected during 2001–2025**, not confirmed pre-2001 planting.
- Age proxy/event year are estimates, not exact planting year.
- Logical QC is not field accuracy.

**Source SHA-256:** `e26621a40fceb264f0d65a93b59197184bdb9a95381e7ce9f7e287f0c8557033`

AOI01 reproducibility references from the validated project history/README:
- rasterized Output 1 boundary at 30 m ≈ **13,472.6502 ha**;
- total six classes ≈ same boundary area (difference ~0 ha);
- logical invalid area = **0 ha**;
- stratified QC samples = **120** (20 per class × 6 classes).


## 0. Environment & Earth Engine authentication

In [ ]:
# OPTIONAL (only if packages are missing):
# %pip install -q earthengine-api folium certifi

import math
import json
import time
from pprint import pprint

try:
    import ssl, certifi
    ssl._create_default_https_context = lambda: ssl.create_default_context(cafile=certifi.where())
except Exception:
    pass

import ee

EE_PROJECT = "palm-oil-503202"

try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    # Normal first-run Earth Engine authentication flow.
    # Do NOT paste OAuth tokens into this notebook manually.
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)

print("Earth Engine initialized")
print("earthengine-api version:", getattr(ee, "__version__", "unknown"))
print("Project:", EE_PROJECT)


In [ ]:
def server_value(obj, label=None):
    """Fetch a small EE object for console-style QA. Avoid on huge rasters/vectors."""
    value = obj.getInfo()
    if label:
        print(f"{label}: {value}")
    else:
        pprint(value)
    return value


def task_summary(tasks):
    """Show batch export tasks created by this notebook."""
    if not tasks:
        print("No export tasks were created.")
        return
    for i, task in enumerate(tasks, 1):
        st = task.status()
        print(f"{i:02d}. {st.get('description')} | state={st.get('state')} | id={task.id}")


def start_tasks(tasks):
    """Start explicitly-created EE batch tasks. Call only after QA."""
    for task in tasks:
        task.start()
        print("Started:", task.status().get("description"), "|", task.id)


## 1. Configuration — edit only this block for another AOI

In [ ]:
CFG = {
    "AOI_ID": "SIAK_AOI01",
    "VERSION_TAG": "V1_0_PILOT",
    "REFERENCE_YEAR": 2025,
    "HISTORICAL_START_YEAR": 2001,
    "HISTORICAL_END_YEAR": 2018,
    "RECENT_START_YEAR": 2019,
    "RECENT_END_YEAR": 2025,
    "SCALE": 30,
    "CRS": "EPSG:32647",
    "BOUNDARY_ASSET": "projects/palm-oil-503202/assets/SIAK_AOI01_PLANTATION_BOUNDARY_WEB_WGS84_2025",
    "LANDSAT_CLEAN_ASSET": "projects/palm-oil-503202/assets/SIAK_AOI01_LANDSAT_HISTORICAL_CLEAN_2001_2018",
    "SENTINEL_REGROWTH_ASSET": "projects/palm-oil-503202/assets/SIAK_AOI01_S2_REGROWTH_STACK_2019_2025",
    "OUTPUT_ASSET_ROOT": "projects/palm-oil-503202/assets/",
    "DRIVE_FOLDER": "OUTPUT2_MASTER_SIAK_AOI01_V1_0_PILOT",
    "SAMPLE_PER_CLASS": 20,
    "RANDOM_SEED": 20260803,
    "EXPORT_INTEGRATED_ASSET": False,
    "EXPORT_FINAL_ASSET": False,
    "EXPORT_QC_POINTS_ASSET": False,
    "EXPORT_WEB_DELIVERY": False,
}

OUTPUT_BASE = f"{CFG['AOI_ID']}_OUTPUT2_TREE_AGE_PROXY_{CFG['VERSION_TAG']}"
INTEGRATED_ASSET_ID = CFG["OUTPUT_ASSET_ROOT"] + OUTPUT_BASE + "_INTEGRATED"
FINAL_ASSET_ID = CFG["OUTPUT_ASSET_ROOT"] + OUTPUT_BASE + "_FINAL_6CLASS"
QC_POINTS_ASSET_ID = CFG["OUTPUT_ASSET_ROOT"] + OUTPUT_BASE + "_QC_POINTS"

START_EXPORT_TASKS = False
print(OUTPUT_BASE)


## 2. Load standardized inputs + boundary mask

In [ ]:
boundary = ee.FeatureCollection(CFG["BOUNDARY_ASSET"])
landsat = ee.Image(CFG["LANDSAT_CLEAN_ASSET"])
sentinel = ee.Image(CFG["SENTINEL_REGROWTH_ASSET"])
geometry = boundary.geometry()
exportRegion = geometry.bounds(1)

print("AOI ID:", CFG["AOI_ID"])
print("Boundary asset:", CFG["BOUNDARY_ASSET"])
print("Landsat clean asset:", CFG["LANDSAT_CLEAN_ASSET"])
print("Sentinel regrowth asset:", CFG["SENTINEL_REGROWTH_ASSET"])
server_value(boundary.size(), "Boundary polygon count")
server_value(landsat.bandNames(), "Available Landsat bands")
server_value(sentinel.bandNames(), "Available Sentinel-2 bands")

REQUIRED_LANDSAT = [
    "CLEAN_EVENT_YEAR", "AGE_PROXY_2025", "CLEAN_AGE_CLASS", "CONFIDENCE_CLASS",
    "PATCH_PIXEL_COUNT", "DISTURBANCE_SCORE", "MAX_RECOVERY_NDVI", "MAX_RECOVERY_NBR",
]
REQUIRED_S2 = ["REGROWTH_STATUS", "DETECTED_YEAR", "REGROWTH_FLAG", "AGE_PROXY_YEARS", "MAX_DECLINE_SCORE"]
print("Required Landsat bands:", REQUIRED_LANDSAT)
print("Required Sentinel-2 bands:", REQUIRED_S2)

boundaryMask = (ee.Image.constant(0).byte().paint(boundary, 1)
                .rename("BOUNDARY_MASK")
                .reproject(crs=CFG["CRS"], scale=CFG["SCALE"])
                .selfMask().clip(geometry))


## 3. Standardize Landsat historical and Sentinel-2 recent evidence

In [ ]:
landsatEventYear = landsat.select("CLEAN_EVENT_YEAR").rename("LANDSAT_EVENT_YEAR").toInt16().clip(geometry)
landsatAgeProxy = landsat.select("AGE_PROXY_2025").rename("LANDSAT_AGE_PROXY_2025").toInt16().clip(geometry)
landsatAgeClass = landsat.select("CLEAN_AGE_CLASS").rename("LANDSAT_AGE_CLASS").toByte().clip(geometry)
landsatConfidence = landsat.select("CONFIDENCE_CLASS").rename("LANDSAT_CONFIDENCE_CLASS").toByte().clip(geometry)
landsatPatchPixelCount = landsat.select("PATCH_PIXEL_COUNT").rename("LANDSAT_PATCH_PIXEL_COUNT").toInt32().clip(geometry)
landsatDisturbanceScore = landsat.select("DISTURBANCE_SCORE").rename("LANDSAT_DISTURBANCE_SCORE").toFloat().clip(geometry)
landsatMaxRecoveryNDVI = landsat.select("MAX_RECOVERY_NDVI").rename("LANDSAT_MAX_RECOVERY_NDVI").toFloat().clip(geometry)
landsatMaxRecoveryNBR = landsat.select("MAX_RECOVERY_NBR").rename("LANDSAT_MAX_RECOVERY_NBR").toFloat().clip(geometry)

landsatValid = (landsatEventYear.unmask(0).gte(CFG["HISTORICAL_START_YEAR"])
    .And(landsatEventYear.unmask(0).lte(CFG["HISTORICAL_END_YEAR"]))
    .And(landsatAgeClass.unmask(0).gt(0)))

s2Status = sentinel.select("REGROWTH_STATUS").rename("S2_REGROWTH_STATUS").toByte().clip(geometry)
s2DetectedYear = sentinel.select("DETECTED_YEAR").rename("S2_DETECTED_YEAR").toInt16().clip(geometry)
s2RegrowthFlag = sentinel.select("REGROWTH_FLAG").rename("S2_REGROWTH_FLAG").toByte().clip(geometry)
s2AgeProxy = sentinel.select("AGE_PROXY_YEARS").rename("S2_AGE_PROXY_YEARS").toInt16().clip(geometry)
s2MaxDeclineScore = sentinel.select("MAX_DECLINE_SCORE").rename("S2_MAX_DECLINE_SCORE").toFloat().clip(geometry)

# S2 status semantics: 0 no strong decline; 1 decline without confirmed regrowth;
# 2 confirmed regrowth candidate; 3 recent event / insufficient evidence.
s2Valid = (s2Status.unmask(0).eq(2)
    .And(s2RegrowthFlag.unmask(0).eq(1))
    .And(s2DetectedYear.unmask(0).gte(CFG["RECENT_START_YEAR"]))
    .And(s2DetectedYear.unmask(0).lte(CFG["RECENT_END_YEAR"])))
s2Uncertain = s2Status.unmask(0).eq(1).Or(s2Status.unmask(0).eq(3))


## 4. Landsat–Sentinel integration (validated 20A–20H logic)

In [ ]:
detectedMask = landsatValid.Or(s2Valid)

combinedEventYearDetected = (landsatEventYear.unmask(0)
    .where(s2Valid, s2DetectedYear.unmask(0))
    .rename("COMBINED_EVENT_YEAR").toInt16().updateMask(detectedMask))
combinedAgeProxyDetected = (ee.Image.constant(CFG["REFERENCE_YEAR"])
    .subtract(combinedEventYearDetected)
    .rename(f"COMBINED_AGE_PROXY_{CFG['REFERENCE_YEAR']}").toInt16().updateMask(detectedMask))

sourceCodeDetected = (ee.Image.constant(0)
    .where(landsatValid, 1)
    .where(s2Valid, 2)
    .where(landsatValid.And(s2Uncertain).And(s2Valid.Not()), 3)
    .rename("SOURCE_CODE").toByte().updateMask(detectedMask))

evidenceCodeDetected = (ee.Image.constant(0)
    .where(landsatValid, 1)
    .where(s2Valid, 2)
    .where(landsatValid.And(s2Uncertain).And(s2Valid.Not()), 3)
    .rename("EVIDENCE_CODE").toByte().updateMask(detectedMask))

qcStatusDetected = (ee.Image.constant(1)
    .where(sourceCodeDetected.eq(3).Or(evidenceCodeDetected.eq(3)).Or(s2Uncertain), 2)
    .rename("QC_STATUS").toByte().updateMask(detectedMask))

integratedStack = ee.Image.cat([
    combinedEventYearDetected, combinedAgeProxyDetected, sourceCodeDetected, evidenceCodeDetected, qcStatusDetected,
    landsatEventYear, landsatAgeProxy, landsatAgeClass, landsatConfidence, landsatPatchPixelCount,
    landsatDisturbanceScore, landsatMaxRecoveryNDVI, landsatMaxRecoveryNBR,
    s2DetectedYear, s2AgeProxy, s2Status, s2RegrowthFlag, s2MaxDeclineScore,
    s2Valid.rename("S2_VALID_FLAG").toByte(), s2Uncertain.rename("S2_UNCERTAIN_FLAG").toByte(),
]).clip(geometry).set({
    "AOI_ID": CFG["AOI_ID"], "OUTPUT_STAGE": "MASTER_INTEGRATED",
    "ANALYSIS_PERIOD": f"{CFG['HISTORICAL_START_YEAR']}-{CFG['RECENT_END_YEAR']}",
    "REFERENCE_YEAR": CFG["REFERENCE_YEAR"], "SECOND_MMU_FILTER": "NOT_APPLIED",
    "SCRIPT": "20I_MASTER_PIPELINE", "VERSION_TAG": CFG["VERSION_TAG"],
})
server_value(integratedStack.bandNames(), "Integrated stack bands")


## 5. Six final age/event classes across the full Output 1 boundary

In [ ]:
finalEventYear = combinedEventYearDetected.unmask(0).rename("FINAL_EVENT_YEAR").toInt16().updateMask(boundaryMask)
eventDetectedFlag = (finalEventYear.gte(CFG["HISTORICAL_START_YEAR"])
    .And(finalEventYear.lte(CFG["RECENT_END_YEAR"]))
    .rename("EVENT_DETECTED_FLAG").toByte().updateMask(boundaryMask))
noEventFlag = (eventDetectedFlag.Not()
    .rename(f"NO_EVENT_DETECTED_{CFG['HISTORICAL_START_YEAR']}_{CFG['RECENT_END_YEAR']}_FLAG")
    .toByte().updateMask(boundaryMask))

finalAgeClass = (ee.Image.constant(0)
    .where(finalEventYear.gte(2001).And(finalEventYear.lte(2005)), 1)
    .where(finalEventYear.gte(2006).And(finalEventYear.lte(2010)), 2)
    .where(finalEventYear.gte(2011).And(finalEventYear.lte(2015)), 3)
    .where(finalEventYear.gte(2016).And(finalEventYear.lte(2020)), 4)
    .where(finalEventYear.gte(2021).And(finalEventYear.lte(2025)), 5)
    .rename("FINAL_AGE_CLASS_6").toByte().updateMask(boundaryMask))

eventAgeProxy = (ee.Image.constant(CFG["REFERENCE_YEAR"]).subtract(finalEventYear)
                 .rename(f"EVENT_AGE_PROXY_{CFG['REFERENCE_YEAR']}").toInt16().updateMask(eventDetectedFlag))
finalSourceCode = sourceCodeDetected.unmask(0).rename("FINAL_SOURCE_CODE").toByte().updateMask(boundaryMask)
finalEvidenceCode = evidenceCodeDetected.unmask(0).rename("FINAL_EVIDENCE_CODE").toByte().updateMask(boundaryMask)
finalQCStatus = qcStatusDetected.unmask(0).rename("FINAL_QC_STATUS").toByte().updateMask(boundaryMask)
finalLandsatConfidence = landsatConfidence.unmask(0).rename("FINAL_LANDSAT_CONFIDENCE").toByte().updateMask(boundaryMask)
finalS2ValidFlag = s2Valid.unmask(0).rename("FINAL_S2_VALID_FLAG").toByte().updateMask(boundaryMask)
finalS2UncertainFlag = s2Uncertain.unmask(0).rename("FINAL_S2_UNCERTAIN_FLAG").toByte().updateMask(boundaryMask)

finalStack = ee.Image.cat([
    finalEventYear, eventAgeProxy, finalAgeClass, eventDetectedFlag, noEventFlag,
    finalSourceCode, finalEvidenceCode, finalQCStatus, finalLandsatConfidence,
    finalS2ValidFlag, finalS2UncertainFlag,
]).clip(geometry).set({
    "AOI_ID": CFG["AOI_ID"], "OUTPUT_ID": "OUTPUT2_FINAL_TREE_AGE_PROXY", "VERSION_TAG": CFG["VERSION_TAG"],
    "REFERENCE_YEAR": CFG["REFERENCE_YEAR"],
    "ANALYSIS_PERIOD": f"{CFG['HISTORICAL_START_YEAR']}-{CFG['RECENT_END_YEAR']}",
    "CLASS_0": "POSSIBLY_PRE_2001_OR_NO_EVENT_DETECTED_2001_2025",
    "CLASS_1": "2001-2005", "CLASS_2": "2006-2010", "CLASS_3": "2011-2015",
    "CLASS_4": "2016-2020", "CLASS_5": "2021-2025",
    "SPATIAL_MASK": "OUTPUT1_PLANTATION_BOUNDARY", "SECOND_MMU_FILTER": "NOT_APPLIED",
    "SPATIAL_RESOLUTION_M": CFG["SCALE"], "CRS": CFG["CRS"],
    "LIMITATION": "Age proxy and event period are estimates, not confirmed planting years. Class 0 means possible pre-2001 plantation or no detected event during 2001-2025.",
    "SCRIPT": "20I_MASTER_PIPELINE",
})
server_value(finalStack.bandNames(), "Final stack bands")


## 6. Area summary + final logical QC

In [ ]:
def calculate_area_ha(mask_image):
    result = (ee.Image.pixelArea().divide(10000).rename("AREA_HA")
              .updateMask(mask_image)
              .reduceRegion(reducer=ee.Reducer.sum(), geometry=geometry,
                            scale=CFG["SCALE"], crs=CFG["CRS"], maxPixels=1e13, tileScale=4))
    return ee.Number(ee.Dictionary(result).get("AREA_HA", 0))

classLabels = ee.Dictionary({
    "0":"POSSIBLY_PRE_2001_OR_NO_EVENT_2001_2025",
    "1":"2001-2005", "2":"2006-2010", "3":"2011-2015", "4":"2016-2020", "5":"2021-2025",
})

totalBoundaryAreaHa = calculate_area_ha(boundaryMask)

def class_summary_row(code):
    code = ee.Number(code)
    classAreaHa = calculate_area_ha(finalAgeClass.eq(code))
    return ee.Feature(None, {
        "AOI_ID": CFG["AOI_ID"], "CLASS_CODE": code,
        "CLASS_LABEL": classLabels.get(code.format("%.0f")), "AREA_HA": classAreaHa,
        "PERCENT_BOUNDARY": classAreaHa.divide(totalBoundaryAreaHa).multiply(100),
        "REFERENCE_YEAR": CFG["REFERENCE_YEAR"],
    })

classSummary = ee.FeatureCollection(ee.List.sequence(0, 5).map(class_summary_row))
totalClassAreaHa = ee.Number(classSummary.aggregate_sum("AREA_HA"))

print("=== 20I CLASS SUMMARY CHECK ===")
server_value(totalBoundaryAreaHa, "Boundary raster area (ha)")
server_value(classSummary, "Six-class summary")
server_value(totalClassAreaHa, "Total six-class area (ha)")
server_value(totalBoundaryAreaHa.subtract(totalClassAreaHa), "Boundary minus class total (ha)")

ageProxyCheck = eventAgeProxy.unmask(-9999)
classYearOK = (finalAgeClass.eq(0).And(finalEventYear.eq(0))
    .Or(finalAgeClass.eq(1).And(finalEventYear.gte(2001)).And(finalEventYear.lte(2005)))
    .Or(finalAgeClass.eq(2).And(finalEventYear.gte(2006)).And(finalEventYear.lte(2010)))
    .Or(finalAgeClass.eq(3).And(finalEventYear.gte(2011)).And(finalEventYear.lte(2015)))
    .Or(finalAgeClass.eq(4).And(finalEventYear.gte(2016)).And(finalEventYear.lte(2020)))
    .Or(finalAgeClass.eq(5).And(finalEventYear.gte(2021)).And(finalEventYear.lte(2025))))

eventFlagOK = eventDetectedFlag.eq(finalAgeClass.gt(0))
noEventFlagOK = noEventFlag.eq(finalAgeClass.eq(0))
expectedAgeProxy = ee.Image.constant(CFG["REFERENCE_YEAR"]).subtract(finalEventYear)
ageProxyOK = finalAgeClass.eq(0).Or(finalAgeClass.gt(0).And(ageProxyCheck.eq(expectedAgeProxy)))
sourceOK = finalAgeClass.eq(0).And(finalSourceCode.eq(0)).Or(finalAgeClass.gt(0).And(finalSourceCode.gte(1)).And(finalSourceCode.lte(3)))
evidenceOK = finalAgeClass.eq(0).And(finalEvidenceCode.eq(0)).Or(finalAgeClass.gt(0).And(finalEvidenceCode.gte(1)).And(finalEvidenceCode.lte(3)))
qcStatusOK = finalAgeClass.eq(0).And(finalQCStatus.eq(0)).Or(finalAgeClass.gt(0).And(finalQCStatus.eq(1).Or(finalQCStatus.eq(2))))

logicalErrorCount = ee.Image.cat([
    classYearOK.Not().rename("ERROR_CLASS_YEAR").toByte(),
    eventFlagOK.Not().rename("ERROR_EVENT_FLAG").toByte(),
    noEventFlagOK.Not().rename("ERROR_NO_EVENT_FLAG").toByte(),
    ageProxyOK.Not().rename("ERROR_AGE_PROXY").toByte(),
    sourceOK.Not().rename("ERROR_SOURCE_CODE").toByte(),
    evidenceOK.Not().rename("ERROR_EVIDENCE_CODE").toByte(),
    qcStatusOK.Not().rename("ERROR_QC_STATUS").toByte(),
]).reduce(ee.Reducer.sum()).rename("FINAL_QC_ERROR_COUNT").toByte().updateMask(boundaryMask)

logicallyValid = logicalErrorCount.eq(0)
logicallyInvalid = logicalErrorCount.gt(0)
logicallyValidAreaHa = calculate_area_ha(logicallyValid)
logicallyInvalidAreaHa = calculate_area_ha(logicallyInvalid)

qcSummary = ee.FeatureCollection([
    ee.Feature(None, {"AOI_ID":CFG["AOI_ID"], "STATUS":"TOTAL_BOUNDARY", "AREA_HA":totalBoundaryAreaHa, "PERCENT_TOTAL":100}),
    ee.Feature(None, {"AOI_ID":CFG["AOI_ID"], "STATUS":"LOGICALLY_VALID", "AREA_HA":logicallyValidAreaHa, "PERCENT_TOTAL":logicallyValidAreaHa.divide(totalBoundaryAreaHa).multiply(100)}),
    ee.Feature(None, {"AOI_ID":CFG["AOI_ID"], "STATUS":"LOGICALLY_INVALID", "AREA_HA":logicallyInvalidAreaHa, "PERCENT_TOTAL":logicallyInvalidAreaHa.divide(totalBoundaryAreaHa).multiply(100)}),
])

print("=== 20I LOGICAL QC ===")
server_value(logicallyValidAreaHa, "Logically valid area (ha)")
server_value(logicallyInvalidAreaHa, "Logically invalid area (ha)")
server_value(qcSummary, "QC summary")


## 7. Stratified QC points (20 × 6 classes = 120 expected)

In [ ]:
samplingImage = (ee.Image.cat([
    finalEventYear.unmask(0), eventAgeProxy.unmask(-9999), finalAgeClass.unmask(0),
    eventDetectedFlag.unmask(0), noEventFlag.unmask(0), finalSourceCode.unmask(0),
    finalEvidenceCode.unmask(0), finalQCStatus.unmask(0), finalLandsatConfidence.unmask(0),
    finalS2ValidFlag.unmask(0), finalS2UncertainFlag.unmask(0), logicalErrorCount.unmask(0),
]).updateMask(boundaryMask).addBands(ee.Image.pixelLonLat()))

qcPoints = samplingImage.stratifiedSample(
    numPoints=CFG["SAMPLE_PER_CLASS"], classBand="FINAL_AGE_CLASS_6", region=geometry,
    scale=CFG["SCALE"], projection=ee.Projection(CFG["CRS"]), seed=CFG["RANDOM_SEED"],
    dropNulls=False, tileScale=4, geometries=True,
)

def enrich_qc_point(feature):
    feature = ee.Feature(feature)
    classCode = ee.Number(feature.get("FINAL_AGE_CLASS_6"))
    return feature.set({
        "AOI_ID": CFG["AOI_ID"], "CLASS_LABEL": classLabels.get(classCode.format("%.0f")),
        "MANUAL_QC": "UNREVIEWED", "VISUAL_INTERPRETATION": "", "REVIEW_NOTE": "",
    })

qcPoints = qcPoints.map(enrich_qc_point)
server_value(qcPoints.size(), "QC sample count")
server_value(qcPoints.aggregate_histogram("FINAL_AGE_CLASS_6"), "QC samples per class")


## 8. Style + metadata tables

In [ ]:
styleSpec = ee.FeatureCollection([
    ee.Feature(None, {"CLASS_CODE":0,"CLASS_LABEL":"Possible pre-2001 or no event detected 2001-2025","HEX_COLOR":"#BDBDBD","RED":189,"GREEN":189,"BLUE":189,"DISPLAY_ORDER":0}),
    ee.Feature(None, {"CLASS_CODE":1,"CLASS_LABEL":"2001-2005","HEX_COLOR":"#54278F","RED":84,"GREEN":39,"BLUE":143,"DISPLAY_ORDER":1}),
    ee.Feature(None, {"CLASS_CODE":2,"CLASS_LABEL":"2006-2010","HEX_COLOR":"#2B8CBE","RED":43,"GREEN":140,"BLUE":190,"DISPLAY_ORDER":2}),
    ee.Feature(None, {"CLASS_CODE":3,"CLASS_LABEL":"2011-2015","HEX_COLOR":"#FEC44F","RED":254,"GREEN":196,"BLUE":79,"DISPLAY_ORDER":3}),
    ee.Feature(None, {"CLASS_CODE":4,"CLASS_LABEL":"2016-2020","HEX_COLOR":"#F03B20","RED":240,"GREEN":59,"BLUE":32,"DISPLAY_ORDER":4}),
    ee.Feature(None, {"CLASS_CODE":5,"CLASS_LABEL":"2021-2025","HEX_COLOR":"#31A354","RED":49,"GREEN":163,"BLUE":84,"DISPLAY_ORDER":5}),
])

metadataTable = ee.FeatureCollection([ee.Feature(None, {
    "PRODUCT_NAME":"Tree Age Estimation / Age Proxy", "AOI_ID":CFG["AOI_ID"], "VERSION_TAG":CFG["VERSION_TAG"],
    "REFERENCE_YEAR":CFG["REFERENCE_YEAR"], "ANALYSIS_PERIOD":f"{CFG['HISTORICAL_START_YEAR']}-{CFG['RECENT_END_YEAR']}",
    "SPATIAL_RESOLUTION_M":CFG["SCALE"], "NATIVE_CRS":CFG["CRS"], "NUMBER_OF_CLASSES":6,
    "TOTAL_BOUNDARY_AREA_HA":totalBoundaryAreaHa, "LOGICALLY_VALID_AREA_HA":logicallyValidAreaHa,
    "LOGICALLY_INVALID_AREA_HA":logicallyInvalidAreaHa, "QC_SAMPLE_COUNT":qcPoints.size(),
    "SECOND_MMU_FILTER":"NOT_APPLIED", "REQUIRED_INPUT_1":"OUTPUT1_PLANTATION_BOUNDARY",
    "REQUIRED_INPUT_2":"LANDSAT_HISTORICAL_CLEAN_STACK", "REQUIRED_INPUT_3":"SENTINEL2_REGROWTH_STACK",
    "LIMITATION":"Age proxy and event period are estimates, not confirmed planting years. Class 0 is possible pre-2001 plantation or no event detected during the analysis period.",
})])


## 9. Optional visualization (Folium, no geemap required)

In [ ]:
def make_ee_folium_map(center, zoom=11):
    """Minimal Earth Engine map viewer without geemap."""
    import folium

    m = folium.Map(location=center, zoom_start=zoom, control_scale=True)

    def add_ee_image(image, vis, name, shown=True, opacity=1.0):
        map_id = ee.Image(image).getMapId(vis)
        folium.TileLayer(
            tiles=map_id["tile_fetcher"].url_format,
            attr="Google Earth Engine",
            name=name,
            overlay=True,
            control=True,
            show=shown,
            opacity=opacity,
        ).add_to(m)

    return m, add_ee_image


In [ ]:
center_xy = geometry.centroid(1).coordinates().getInfo()
center = [center_xy[1], center_xy[0]]
Map, add_ee_image = make_ee_folium_map(center, zoom=12)

add_ee_image(finalAgeClass, {"min":0,"max":5,"palette":["BDBDBD","54278F","2B8CBE","FEC44F","F03B20","31A354"]}, "Final 6 Tree Age Proxy Classes", shown=True)
add_ee_image(logicallyInvalid.selfMask(), {"palette":["FF00FF"]}, "Logical Error", shown=False)
add_ee_image(qcPoints.style(color="FFFFFF", fillColor="000000", pointSize=4, width=1), {}, "QC Sample Points", shown=False)
add_ee_image(boundary.style(color="000000", fillColor="00000000", width=1), {}, "Output 1 Boundary", shown=True)

import folium
folium.LayerControl(collapsed=False).add_to(Map)
Map


## 10. Optional asset + web-delivery exports

In [ ]:
EXPORT_TASKS = []

if CFG["EXPORT_INTEGRATED_ASSET"]:
    EXPORT_TASKS.append(ee.batch.Export.image.toAsset(
        image=integratedStack, description=OUTPUT_BASE + "_INTEGRATED_ASSET",
        assetId=INTEGRATED_ASSET_ID, region=geometry, scale=CFG["SCALE"], crs=CFG["CRS"], maxPixels=1e13,
    ))

if CFG["EXPORT_FINAL_ASSET"]:
    EXPORT_TASKS.append(ee.batch.Export.image.toAsset(
        image=finalStack.toInt16(), description=OUTPUT_BASE + "_FINAL_ASSET",
        assetId=FINAL_ASSET_ID, region=geometry, scale=CFG["SCALE"], crs=CFG["CRS"], maxPixels=1e13,
    ))

if CFG["EXPORT_QC_POINTS_ASSET"]:
    EXPORT_TASKS.append(ee.batch.Export.table.toAsset(
        collection=qcPoints, description=OUTPUT_BASE + "_QC_POINTS_ASSET", assetId=QC_POINTS_ASSET_ID,
    ))

if CFG["EXPORT_WEB_DELIVERY"]:
    webAgeClass = finalAgeClass.unmask(255).rename("TREE_AGE_CLASS").toUint8().clip(exportRegion)
    webEventYear = finalEventYear.unmask(-9999).rename("EVENT_YEAR").toInt16().clip(exportRegion)
    webAgeProxy = eventAgeProxy.unmask(-9999).rename(f"AGE_PROXY_{CFG['REFERENCE_YEAR']}").toInt16().clip(exportRegion)
    webFullStack = finalStack.toInt16().unmask(-9999).clip(exportRegion)

    def drive_image(image, name, nodata):
        return ee.batch.Export.image.toDrive(
            image=image, description=name, folder=CFG["DRIVE_FOLDER"], fileNamePrefix=name,
            region=exportRegion, scale=CFG["SCALE"], crs=CFG["CRS"], maxPixels=1e13,
            fileFormat="GeoTIFF", formatOptions={"cloudOptimized": True, "noData": nodata},
        )

    EXPORT_TASKS += [
        drive_image(webAgeClass, f"{CFG['AOI_ID']}_TREE_AGE_CLASS_6_{CFG['REFERENCE_YEAR']}", 255),
        drive_image(webEventYear, f"{CFG['AOI_ID']}_EVENT_YEAR_{CFG['HISTORICAL_START_YEAR']}_{CFG['RECENT_END_YEAR']}", -9999),
        drive_image(webAgeProxy, f"{CFG['AOI_ID']}_EVENT_AGE_PROXY_{CFG['REFERENCE_YEAR']}", -9999),
        drive_image(webFullStack, f"{CFG['AOI_ID']}_TREE_AGE_FULL_STACK_{CFG['REFERENCE_YEAR']}", -9999),
        ee.batch.Export.table.toDrive(collection=classSummary, description=f"{CFG['AOI_ID']}_TREE_AGE_CLASS_SUMMARY_{CFG['REFERENCE_YEAR']}", folder=CFG["DRIVE_FOLDER"], fileNamePrefix=f"{CFG['AOI_ID']}_TREE_AGE_CLASS_SUMMARY_{CFG['REFERENCE_YEAR']}", fileFormat="CSV"),
        ee.batch.Export.table.toDrive(collection=qcSummary, description=f"{CFG['AOI_ID']}_TREE_AGE_QC_SUMMARY_{CFG['REFERENCE_YEAR']}", folder=CFG["DRIVE_FOLDER"], fileNamePrefix=f"{CFG['AOI_ID']}_TREE_AGE_QC_SUMMARY_{CFG['REFERENCE_YEAR']}", fileFormat="CSV"),
        ee.batch.Export.table.toDrive(collection=qcPoints, description=f"{CFG['AOI_ID']}_TREE_AGE_QC_POINTS_{CFG['REFERENCE_YEAR']}", folder=CFG["DRIVE_FOLDER"], fileNamePrefix=f"{CFG['AOI_ID']}_TREE_AGE_QC_POINTS_{CFG['REFERENCE_YEAR']}", fileFormat="CSV"),
        ee.batch.Export.table.toDrive(collection=styleSpec, description=f"{CFG['AOI_ID']}_TREE_AGE_STYLE_SPEC_{CFG['REFERENCE_YEAR']}", folder=CFG["DRIVE_FOLDER"], fileNamePrefix=f"{CFG['AOI_ID']}_TREE_AGE_STYLE_SPEC_{CFG['REFERENCE_YEAR']}", fileFormat="CSV"),
        ee.batch.Export.table.toDrive(collection=metadataTable, description=f"{CFG['AOI_ID']}_TREE_AGE_METADATA_{CFG['REFERENCE_YEAR']}", folder=CFG["DRIVE_FOLDER"], fileNamePrefix=f"{CFG['AOI_ID']}_TREE_AGE_METADATA_{CFG['REFERENCE_YEAR']}", fileFormat="CSV"),
    ]

    boundaryWGS84 = boundary.map(lambda feature: ee.Feature(
        ee.Feature(feature).geometry().transform("EPSG:4326", 1), ee.Feature(feature).toDictionary()
    ))
    EXPORT_TASKS.append(ee.batch.Export.table.toDrive(
        collection=boundaryWGS84,
        description=f"{CFG['AOI_ID']}_PLANTATION_BOUNDARY_WEB_WGS84_{CFG['REFERENCE_YEAR']}",
        folder=CFG["DRIVE_FOLDER"], fileNamePrefix=f"{CFG['AOI_ID']}_PLANTATION_BOUNDARY_WEB_WGS84_{CFG['REFERENCE_YEAR']}",
        fileFormat="GeoJSON",
    ))

task_summary(EXPORT_TASKS)
if START_EXPORT_TASKS:
    start_tasks(EXPORT_TASKS)
else:
    print("Exports NOT started. After QC, run: start_tasks(EXPORT_TASKS)")


## 11. AOI01 reproducibility checkpoint
For the first mentor/team comparison, leave all `EXPORT_* = False` and run from top to bottom.

Expected reference behavior from the validated 20I project:
- Boundary raster area at 30 m ≈ **13,472.65 ha**.
- Logical invalid area = **0 ha**.
- QC sample count = **120**.
- Six classes cover the entire Output 1 boundary.
- No second MMU filtering.

Compare Python vs GEE on:
1. input band lists;
2. integrated/final stack band names;
3. boundary area and six-class area table;
4. logical QC valid/invalid area;
5. sample count per class;
6. map class pattern;
7. export names and metadata.

**Do not interpret logical reproducibility as field accuracy.** Age/event values remain proxy estimates until independent/field validation is available.
